<!-- Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. -->

# MJX 07 — GPU RL with MuJoCo Playground (PPO control)

We train a control policy with **PPO** on the GPU using **MuJoCo Playground**
(DeepMind's GPU-accelerated robot-learning library, built on MJX). The task is
`CartpoleBalance` from the DM Control Suite — a classic control benchmark that
learns in a couple of minutes on the AMD GPU.

## Stable settings for this ROCm stack
Playground imports cleanly on the **torch-free** `auplc-mujoco-mjx` image. Two
knobs keep training healthy on the gfx1151 APU:
- **`num_envs = 256`** — large batches (2048+) trip a GPU memory-aperture fault.
- **`jax_default_matmul_precision = "highest"`** — avoids fp cast overflow.
- **`learning_rate = 3e-4`** (below the 1e-3 default) — at the small `num_envs` the default rate makes PPO diverge; the lower rate climbs smoothly to ~1000 and holds.

We also keep the **best checkpoint** seen during training (not just the final weights) so the rendered policy is always a good one.

Playground physics is forced to the pure-JAX path (`impl="jax"`) because the
`mujoco_warp` kernels are CUDA-only.

In [ ]:
import os, functools
os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jp
# Brax 0.14 calls jax.device_put_replicated, removed in the jax 0.10 that ships
# with jax-rocm. Re-add a single-device version.
if not hasattr(jax, "device_put_replicated"):
    def _dpr(x, devices=None):
        n = len(devices) if devices is not None else jax.local_device_count()
        return jax.tree_util.tree_map(
            lambda a: jax.device_put(jp.broadcast_to(jp.asarray(a)[None], (n,) + jp.asarray(a).shape)), x)
    jax.device_put_replicated = _dpr
# Use the highest matmul precision to avoid fp overflow during training.
jax.config.update("jax_default_matmul_precision", "highest")

import numpy as np
import matplotlib.pyplot as plt
import imageio
from mujoco_playground import registry, wrapper
from mujoco_playground.config import dm_control_suite_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from IPython.display import Video

print("JAX devices:", jax.devices())

In [ ]:
ENV_NAME = "CartpoleBalance"
env = registry.load(ENV_NAME, config_overrides={"impl": "jax"})

# Start from Playground's tuned PPO config, then shrink batch + horizon so the
# notebook trains quickly and safely on this GPU.
cfg = dm_control_suite_params.brax_ppo_config(ENV_NAME)
ppo_params = cfg.to_dict()
net_cfg = ppo_params.pop("network_factory", None)
ppo_params.update(
    num_timesteps=1_500_000,
    num_envs=256,
    batch_size=256,
    num_minibatches=8,
    num_evals=10,
    learning_rate=3e-4,  # lower than the 1e-3 default: small-batch PPO diverges at the high rate
)
print({k: ppo_params[k] for k in ["num_timesteps", "num_envs", "batch_size", "episode_length"]})

In [ ]:
progress = []
def progress_fn(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    progress.append((int(step), r))
    print(f"step {int(step):>9}  eval reward {r:8.1f}")

# Snapshot the policy at every eval so we can keep the BEST one, not just the
# final weights (a safety net against any late-training dip).
checkpoints = []
def policy_params_fn(step, make_policy, params):
    checkpoints.append(jax.tree_util.tree_map(np.array, params))

train_fn = functools.partial(ppo.train, **ppo_params)
if net_cfg:
    train_fn = functools.partial(train_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))

make_inference_fn, params, _ = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    policy_params_fn=policy_params_fn,
    seed=0,
)

eval_rewards = [r for _, r in progress][-len(checkpoints):]
best_idx = int(np.argmax(eval_rewards))
best_params = checkpoints[best_idx]
print(f"training done — best eval reward {eval_rewards[best_idx]:.1f} (final {progress[-1][1]:.1f})")

In [ ]:
steps, rewards = zip(*progress)
plt.figure(figsize=(7, 3))
plt.plot(steps, rewards, marker="o")
plt.xlabel("environment steps"); plt.ylabel("eval episode reward")
plt.title(f"PPO learning curve ({ENV_NAME})"); plt.grid(True); plt.show()

## Did it actually learn? — random vs trained

The learning curve should climb toward ~1000 (the pole stays balanced for the
full episode). To *see* the difference we render two rollouts side by side for
the same 200 steps:

**Left: random actions (not learned)** — the pole tips over and the cart drifts.
**Right: trained PPO policy (learned)** — the pole is held upright.

In [ ]:
# Compare an UNTRAINED (random) policy with the TRAINED PPO policy.
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(best_params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

def rollout(action_fn, n=200, seed=1):
    rng = jax.random.PRNGKey(seed)
    state = reset(rng)
    traj = [state]
    for _ in range(n):  # no early stop, so a failing policy is fully visible
        rng, k = jax.random.split(rng)
        state = step(state, action_fn(state.obs, k))
        traj.append(state)
    return traj

# "Not learned": random actions in the action space.
random_fn  = lambda obs, k: jax.random.uniform(k, (env.action_size,), minval=-1.0, maxval=1.0)
# "Learned": the trained PPO policy.
trained_fn = lambda obs, k: inference(obs, k)[0]

f_rand    = np.asarray(env.render(rollout(random_fn),  height=240, width=320))
f_trained = np.asarray(env.render(rollout(trained_fn), height=240, width=320))
sep = np.full((f_rand.shape[0], f_rand.shape[1], 4, 3), 255, np.uint8)  # white divider
combo = np.concatenate([f_rand, sep, f_trained], axis=2)  # left = random, right = trained
out = "output/videos/mjx07_random_vs_trained.mp4"
imageio.mimsave(out, list(combo), fps=30)
print("saved", combo.shape[0], "frames ->", out)

In [ ]:
Video(url="output/videos/mjx07_random_vs_trained.mp4")